# QLoRA JSON Extraction: End-to-End Training Pipeline

**Stages:**
1. Install & Setup
2. Clone / Sync Project
3. Generate Datasets
4. SFT Training (QLoRA, 4-bit)
5. DPO Alignment
6. Merge Weights
7. GGUF Export

> Run cells top to bottom. Requires a T4 GPU runtime.

---
## Cell 1: Install Dependencies
Installs the exact packages needed. No version pinning — uses whatever Colab has, which is TRL 1.7.x.

In [ ]:
!pip install -q \
    "transformers>=4.44" \
    "trl>=1.7" \
    "peft>=0.12" \
    "bitsandbytes>=0.43" \
    "datasets>=2.18" \
    "accelerate>=0.27" \
    pyyaml faker pydantic

import trl, transformers, peft
print(f"TRL:          {trl.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"PEFT:         {peft.__version__}")

---
## Cell 2: Clone / Sync Project from GitHub
Safe to run multiple times. First run clones. Subsequent runs do a force-sync to latest commit.

In [ ]:
import os
REPO_DIR = "/content/json-extract-qlora-dpo"
REPO_URL = "https://github.com/Tahleels/json-extract-qlora-dpo.git"

if not os.path.exists(REPO_DIR):
    print("First run: Cloning repository...")
    %cd /content
    !git clone {REPO_URL}
else:
    print("Repository exists. Force-syncing latest from GitHub...")
    %cd {REPO_DIR}
    !git fetch origin
    !git reset --hard origin/main

%cd {REPO_DIR}
print(f"Working directory: {os.getcwd()}")

---
## Cell 3: Generate Synthetic Datasets
Creates `data/sft_train.jsonl`, `data/sft_val.jsonl`, `data/sft_test.jsonl`, and `data/dpo_train.jsonl`.

In [ ]:
!python -m src.data.build_sft
!python -m src.data.build_dpo

import os
for f in ["data/sft_train.jsonl", "data/sft_val.jsonl", "data/dpo_train.jsonl"]:
    size = os.path.getsize(f)
    print(f"  {f}: {size:,} bytes")
print("All datasets generated successfully!")

---
## Cell 4: SFT Training (QLoRA, 4-bit NF4)

**What happens here:**
- Load `Qwen2.5-0.5B-Instruct` in 4-bit NF4 quantization using bitsandbytes
- Attach LoRA adapters (rank=16) to attention layers via `get_peft_model()`
- Fine-tune on 3,400 JSON extraction examples using TRL 1.7's `SFTTrainer`
- Save the trained adapter to `artifacts/adapter_sft`

> **TRL 1.7 change:** `peft_config` was removed from `SFTTrainer`. You must wrap the model with `get_peft_model()` first.

In [ ]:
import os, torch, yaml
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from src.config import load_config

# --- Load configs ---
cfg = load_config()
with open("configs/sft.yaml", "r") as f:
    sft_cfg = yaml.safe_load(f)

# --- 4-bit Quantization Config (NF4) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# --- Load Tokenizer ---
print(f"Loading tokenizer: {cfg.base_model}")
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# --- Load Base Model in 4-bit ---
print(f"Loading model in 4-bit NF4: {cfg.base_model}")
model = AutoModelForCausalLM.from_pretrained(
    cfg.base_model,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# --- Wrap model with LoRA adapter (TRL 1.7: must do this BEFORE SFTTrainer) ---
peft_config = LoraConfig(
    r=sft_cfg["lora"]["r"],
    lora_alpha=sft_cfg["lora"]["alpha"],
    lora_dropout=sft_cfg["lora"]["dropout"],
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=sft_cfg["lora"]["target_modules"],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# --- Load & Format Dataset ---
dataset = load_dataset(
    "json",
    data_files={
        "train": "data/sft_train.jsonl",
        "val":   "data/sft_val.jsonl",
    }
)

def format_chat(example):
    messages = [
        {"role": "system",    "content": cfg.system_prompt},
        {"role": "user",      "content": example["input_text"]},
        {"role": "assistant", "content": example["output_json"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_ds = dataset["train"].map(format_chat, remove_columns=dataset["train"].column_names)
val_ds   = dataset["val"].map(format_chat,   remove_columns=dataset["val"].column_names)
print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

# --- SFT Trainer (TRL 1.7 API) ---
# dataset_text_field and max_seq_length live in SFTConfig
# peft_config is NOT passed to SFTTrainer — model is already wrapped above
sft_config = SFTConfig(
    output_dir=sft_cfg["training"]["output_dir"],
    num_train_epochs=sft_cfg["training"]["epochs"],
    per_device_train_batch_size=sft_cfg["training"]["batch_size"],
    gradient_accumulation_steps=sft_cfg["training"]["grad_accum"],
    learning_rate=float(sft_cfg["training"]["learning_rate"]),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="epoch",
    fp16=True,
    dataset_text_field="text",
    max_seq_length=sft_cfg["training"]["max_seq_length"],
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

print("Starting SFT training...")
trainer.train()

os.makedirs("artifacts", exist_ok=True)
trainer.model.save_pretrained("artifacts/adapter_sft")
tokenizer.save_pretrained("artifacts/adapter_sft")
print("SFT adapter saved to: artifacts/adapter_sft")

---
## Cell 5: DPO Alignment (Preference Tuning)

**What happens here:**
- Reload base model in 4-bit and mount the SFT adapter on top
- Run DPO with chosen (clean JSON) vs rejected (corrupted JSON) pairs
- Teaches the model to **never** output markdown fences, preambles, or trailing commas
- Save the aligned adapter to `artifacts/adapter_dpo`

> **TRL 1.7 change:** `tokenizer` arg renamed to `processing_class`. `ref_model=None` works when using a PEFT model (implicit reference).

In [ ]:
import os, torch, yaml
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training
from trl import DPOTrainer, DPOConfig
from src.config import load_config

# --- Load configs ---
cfg = load_config()
with open("configs/dpo.yaml", "r") as f:
    dpo_cfg = yaml.safe_load(f)

# --- 4-bit Quantization Config ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# --- Load Tokenizer ---
print(f"Loading tokenizer: {cfg.base_model}")
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # DPO needs left-padding

# --- Load Base Model in 4-bit ---
print(f"Loading base model in 4-bit: {cfg.base_model}")
base_model = AutoModelForCausalLM.from_pretrained(
    cfg.base_model,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model = prepare_model_for_kbit_training(base_model)

# --- Load SFT adapter as trainable for DPO ---
print("Loading SFT adapter for DPO...")
model = PeftModel.from_pretrained(
    base_model,
    "artifacts/adapter_sft",
    is_trainable=True,
)

# --- Load & Format DPO Dataset ---
# TRL 1.7 DPOTrainer expects columns: prompt, chosen, rejected
raw_dataset = load_dataset("json", data_files={"train": "data/dpo_train.jsonl"})["train"]

def format_dpo(example):
    messages = [
        {"role": "system", "content": cfg.system_prompt},
        {"role": "user",   "content": example["prompt"]},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return {
        "prompt":   prompt_text,
        "chosen":   example["chosen"]   + tokenizer.eos_token,
        "rejected": example["rejected"] + tokenizer.eos_token,
    }

dpo_dataset = raw_dataset.map(format_dpo, remove_columns=raw_dataset.column_names)
print(f"DPO pairs: {len(dpo_dataset)}")

# --- DPO Trainer (TRL 1.7 API) ---
# 'tokenizer' param was renamed to 'processing_class' in TRL 1.7
dpo_config = DPOConfig(
    output_dir=dpo_cfg["training"]["output_dir"],
    beta=dpo_cfg["training"]["beta"],
    num_train_epochs=dpo_cfg["training"]["epochs"],
    per_device_train_batch_size=dpo_cfg["training"]["batch_size"],
    gradient_accumulation_steps=dpo_cfg["training"]["grad_accum"],
    learning_rate=float(dpo_cfg["training"]["learning_rate"]),
    max_length=dpo_cfg["training"]["max_seq_length"],
    max_prompt_length=256,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    report_to="none",
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,          # None = use the frozen PEFT base as implicit reference
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,  # renamed from 'tokenizer' in TRL 1.7
)

print("Starting DPO alignment...")
trainer.train()

trainer.model.save_pretrained("artifacts/adapter_dpo")
tokenizer.save_pretrained("artifacts/adapter_dpo")
print("DPO adapter saved to: artifacts/adapter_dpo")

---
## Cell 6: Merge LoRA Weights

**What happens here:**
- Load the base model in full FP16 (no quantization — needed for a clean merge)
- Apply the DPO adapter on top
- Call `merge_and_unload()` to fold adapter weights permanently into the base model
- Save the resulting standalone model to `artifacts/merged_model`

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.config import load_config

cfg = load_config()

print(f"Loading base model in FP16 (no quantization): {cfg.base_model}")
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    cfg.base_model,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print("Applying DPO adapter...")
model = PeftModel.from_pretrained(base_model, "artifacts/adapter_dpo")

print("Merging adapter weights into base model...")
merged_model = model.merge_and_unload()

print("Saving merged model...")
merged_model.save_pretrained("artifacts/merged_model")
tokenizer.save_pretrained("artifacts/merged_model")
print("Done! Merged model saved to: artifacts/merged_model")

---
## Cell 7: GGUF Export & Quantization

**What happens here:**
- Clone and compile `llama.cpp` using cmake
- Convert the merged HF model to GGUF FP16 format
- Quantize to `Q4_K_M` (~400MB) — ready to run locally on a CPU laptop

In [ ]:
import os

# Clone and build llama.cpp
if not os.path.exists("llama.cpp"):
    !git clone https://github.com/ggerganov/llama.cpp.git
!cmake llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF
!cmake --build llama.cpp/build --config Release -j$(nproc)
!pip install -q -r llama.cpp/requirements.txt

# Convert HF -> GGUF FP16
!python llama.cpp/convert_hf_to_gguf.py \
    artifacts/merged_model \
    --outfile artifacts/model-f16.gguf \
    --outtype f16

# Quantize to Q4_K_M
!./llama.cpp/build/bin/llama-quantize \
    artifacts/model-f16.gguf \
    artifacts/model-Q4_K_M.gguf \
    Q4_K_M

size_mb = os.path.getsize("artifacts/model-Q4_K_M.gguf") / 1024 / 1024
print(f"GGUF model ready: artifacts/model-Q4_K_M.gguf ({size_mb:.1f} MB)")